In [ ]:
BRANCH = "main"

%cd /tmp
!rm -rf damage-detection
!git clone -b {BRANCH} https://github.com/azrairemd/damage-detection.git
%cd damage-detection
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
import os
import shutil
import time
import torch
from torch.utils.data import DataLoader, Subset

from src.data.dataset import XBDDataset
from src.losses.combined_loss import CombinedLoss
from src.models.siamese_unet import SiameseUNet
from src.train import fit, train_one_epoch, validate
from src.utils.config import load_config

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

## Testing the Siamese U-NET model with fake data

In [ ]:
pre = torch.randn(2, 3, 512, 512)
post = torch.randn(2, 3, 512, 512)

for fusion in ["concat", "diff"]:
    model = SiameseUNet(fusion=fusion)
    n_params = sum(p.numel() for p in model.parameters())
    with torch.no_grad():
        logits = model(pre, post)
    print(f"{fusion}: output {tuple(logits.shape)}, {n_params / 1e6:.1f} million parameters")

**Model check** Both fusion methods produce an output of shape (batch, 5, H, W), the same as the baseline.

# Config Paths 
**config path for whichever model version is running**

## Question 1: concat or diff? 

### 1. siamese - concat

In [ ]:
# CONFIG_PATH = "configs/siamese_concat.yaml"   # done: val score 0.5981

### 2. siamese - diff

In [ ]:
# CONFIG_PATH = "configs/siamese_diff.yaml" # done: 0.5865

## Question 2: which loss function (cross entropy | focal loss | focal loss + dice)?

### 1. cross entropy (siamese (concat))

In [ ]:
# CONFIG_PATH = "configs/siamese_ce.yaml" # done: 0.7079

### 2. focal loss (siamese (concat))

In [ ]:
CONFIG_PATH = "configs/siamese_focal.yaml"

## Setup for all tests

In [ ]:
config = load_config(CONFIG_PATH)
data = config["data"]
train_cfg = config["training"]
loss_cfg = config["loss"]
model_cfg = config["model"]

In [ ]:
# data
train_ds = XBDDataset(data["train_split"], data["xbd_root"], data["mask_root"],
                      train=True, crop_size=data["crop_size"])
val_ds = XBDDataset(data["val_split"], data["xbd_root"], data["mask_root"], train=False)

train_loader = DataLoader(train_ds, batch_size=train_cfg["batch_size"], shuffle=True,
                          num_workers=train_cfg["num_workers"], drop_last=True)
val_loader = DataLoader(val_ds, batch_size=train_cfg["val_batch_size"], shuffle=False,
                        num_workers=train_cfg["num_workers"])

# model, loss and optimizer
model = SiameseUNet(encoder=model_cfg["encoder"], fusion=model_cfg["fusion"]).to(device)
loss_fn = CombinedLoss(alpha=torch.tensor(loss_cfg["class_weights"]),
                       gamma=loss_cfg["gamma"], lam=loss_cfg["lam"])
optimizer = torch.optim.AdamW(model.parameters(), lr=train_cfg["learning_rate"])

## 1. Siamese model - concat 

### 1.2 Full training

In [ ]:
# out_dir = train_cfg["out_dir"]
# os.makedirs(out_dir, exist_ok=True)
# shutil.copy(CONFIG_PATH, os.path.join(out_dir, "config.yaml"))

# start = time.time()
# best = fit(model, train_loader, val_loader, loss_fn, optimizer, device,
#            max_epochs=train_cfg["max_epochs"], out_dir=out_dir)

# print(f"Best val score: {best:.4f}")
# print(f"Total time: {(time.time() - start) / 3600:.1f} hours")

**Siamese concat results**
- Best validation xView2 score: **0.5981** at epoch 13 (localization F1 0.787, damage F1 0.517); early stopping at epoch 17.
- The fluctuations of the validation score now come from **destroyed**, not minor. A confusion matrix (Phase 7) is needed to confirm which one.
- Given the strong fluctuations and the unfixed seeds, the difference of 0.011 in the overall score is not meaningful.

Using the *pre-disaster* image clearly improves building **localization (+0.07 F1)** and the recognition of **destroyed buildings (+0.11 F1)**, but **reduces the minor-damage F1 (−0.09)**. 

Because the damage F1 is a harmonic mean, the overall *xView2 score* is similar for **both models (0.598 vs. 0.610)**, and the difference is smaller than the epoch-to-epoch fluctuations.

**Comparison with the baseline (validation set, best epoch)**

| | Baseline (post only) | Siamese (concat) | Difference |
|---|---|---|---|
| **xView2 score** | **0.6095** | 0.5981 | −0.011 |
| Localization F1 | 0.717 | **0.787** | **+0.070** |
| Damage F1 (harmonic mean) | **0.563** | 0.517 | −0.046 |
| No damage F1 | 0.826 | **0.849** | +0.023 |
| Minor damage F1 | **0.371** | 0.280 | **−0.091** |
| Major damage F1 | **0.611** | 0.603 | −0.008 |
| Destroyed F1 | 0.644 | **0.753** | **+0.109** |
| Best epoch / stopped at | 11 / 15 | 13 / 17 | |
| Training time | 2.2 h | 2.6 h | |
| Time per epoch | ~9 min | ~9 min | |
| Parameters | 24.4 M | 26.4 M | |

Bold: the better model for each row.

## 2. Siamese model - diff

### Full training

In [ ]:
# out_dir = train_cfg["out_dir"]
# os.makedirs(out_dir, exist_ok=True)
# shutil.copy(CONFIG_PATH, os.path.join(out_dir, "config.yaml"))

# start = time.time()
# best = fit(model, train_loader, val_loader, loss_fn, optimizer, device,
#           max_epochs=train_cfg["max_epochs"], out_dir=out_dir)

# print(f"Best val score: {best:.4f}")
# print(f"Total time: {(time.time() - start) / 3600:.1f} hours")

**Siamese - diff results and A2 conclusion**

| | Baseline | Siamese concat | Siamese diff |
|---|---|---|---|
| **xView2 score** | **0.6095** | 0.5981 | 0.5865 |
| Localization F1 | 0.717 | 0.787 | **0.791** |
| Damage F1 | **0.563** | 0.517 | 0.499 |
| No damage | 0.826 | 0.849 | **0.862** |
| Minor | **0.371** | 0.280 | 0.267 |
| Major | **0.611** | 0.603 | 0.568 |
| Destroyed | 0.644 | **0.753** | 0.743 |
| Best epoch / stopped at | 11 / 15 | 13 / 17 | 6 / 10 |
| Mean of the 3 best epochs | **0.595** | 0.579 | 0.554 |

- **A2:** *concatenation fusion* is slightly better than difference fusion (0.598 vs 0.587), so concat is **kept for the next experiments**.
- **Destroyed collapse:** in both Siamese models, the destroyed F1 drops to ~0 in some epochs (concat: 11, 12, 17; diff: 7, 8, 10) while the localization F1 stays high. This never happens in the baseline. These collapses fill the patience counter and stop training early. To be investigated with the confusion matrix in Phase 7.

## **---------------------------------------------**

### 1. cross entropy (siamese (concat))

In [ ]:
# out_dir = train_cfg["out_dir"]
# os.makedirs(out_dir, exist_ok=True)
# shutil.copy(CONFIG_PATH, os.path.join(out_dir, "config.yaml"))

# start = time.time()
# best = fit(model, train_loader, val_loader, loss_fn, optimizer, device,
#            max_epochs=train_cfg["max_epochs"], out_dir=out_dir)

# print(f"Best val score: {best:.4f}")
# print(f"Total time: {(time.time() - start) / 3600:.1f} hours")

**Siamese (concat) + class-weighted CE results**

| Model | Loss | Val score | Loc F1 | Dmg F1 | Best epoch / stopped at |
|---|---|---|---|---|---|
| Baseline | Focal + Dice | 0.6095 | 0.717 | 0.563 | 11 / 15 |
| Siamese concat | Focal + Dice | 0.5981 | 0.787 | 0.517 | 13 / 17 |
| Siamese diff | Focal + Dice | 0.5865 | 0.791 | 0.499 | 6 / 10 |
| **Siamese concat** | **CE** | **0.7079** | 0.788 | **0.674** | **20 / 20** |

- With class-weighted cross-entropy (focal γ = 0, no Dice), the Siamese model reaches **0.7079**, the best score so far: **+0.11** compared to the same model trained with Focal + Dice, and **+0.10** compared to the baseline.
- The gain comes entirely from the damage F1 (0.517 → 0.674); localization stays the same (~0.79).
- Training reached `max_epochs` (20) with a new best score at the last epoch, so the model **was still improving**. All experiments use the same limit of 20 epochs, so the comparison remains *fair*.
- **A1 revisited:** with Focal + Dice, *the pre-disaster image did not improve the overall score*; with CE, the Siamese model is clearly better than the baseline. 

### 3. focal loss (siamese (concat))

In [ ]:
out_dir = train_cfg["out_dir"]
os.makedirs(out_dir, exist_ok=True)
shutil.copy(CONFIG_PATH, os.path.join(out_dir, "config.yaml"))

start = time.time()
best = fit(model, train_loader, val_loader, loss_fn, optimizer, device,
           max_epochs=train_cfg["max_epochs"], out_dir=out_dir)

print(f"Best val score: {best:.4f}")
print(f"Total time: {(time.time() - start) / 3600:.1f} hours")

**Focal-only results**

| Loss (Siamese concat) | Val score | Loc F1 | Dmg F1 | Minor | Major | Destroyed | Best / stopped |
|---|---|---|---|---|---|---|---|
| Focal + Dice | 0.5981 | 0.787 | 0.517 | 0.280 | 0.603 | 0.753 | 13 / 17 |
| Focal only | 0.6583 | 0.790 | 0.602 | 0.365 | 0.637 | 0.819 | 19 / 20 |
| **Class-weighted CE** | **0.7079** | 0.788 | **0.674** | | | | 20 / 20 |

- **CE > Focal only > Focal + Dice.** Removing the Dice loss improves the score by +0.06, and removing the focal term (γ = 2 → 0) by another +0.05.
- **Final model:** Siamese U-Net (concat) trained with class-weighted cross-entropy (val score 0.7079), selected on the validation set only.